# AusGrid2013: current deduplicated experiment results (360 nodes)

Use the completed ordinary FHF, HFF, and HFFPS experiments in this project's `fhf_dedup_ausgrid2013/` directory.
Show accuracy, training records, timing, and hierarchy consistency on the current deduplicated data. Include one shared Naive baseline.
Rebuild all statistics from the current per-node metrics, forecast CSVs, and training residual files, without starting model training.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from PPHF_modules.results_check.current_results import (
    prepare_current_results, core_accuracy_table, plot_training, plot_timing,
)

In [ ]:
DATASET = 'ausgrid2013'
SOURCE_DIR = ROOT / 'fhf_dedup_ausgrid2013'
OUTPUT_DIR = ROOT / 'Results_check' / 'deduplication_results' / DATASET
RECOMPUTE = False  # Rebuild automatically when input contents change. True forces recomputation.
print('Source:', SOURCE_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
tables = prepare_current_results(DATASET, root=ROOT, source_dir=SOURCE_DIR,
                                 output_dir=OUTPUT_DIR, force=RECOMPUTE)
display(tables['hierarchy_audit'])

## Accuracy comparison

Compute MAE, MASE, and RMSSE per node, then average equally within each semantic evaluation level.
Series that become leaves after deduplication belong to the bottom level. Retain structural depth for subsequent parent-child constraint checks.
The core table includes base, BU, MinT-shrinkage, WLSv, and the shared Naive baseline. See `accuracy_by_level.csv` for all methods.

In [ ]:
core = core_accuracy_table(tables)
display(core)
core.to_csv(OUTPUT_DIR / 'core_accuracy_by_level.csv')

## All methods and MCB inputs

The table below retains all reconciliation methods. Pair MCB inputs by forecast horizon and node in the two CSVs,
`mcb_mase.csv` and `mcb_rmsse.csv`. MCB uses base, BU, MinT-shrinkage, and WLSv for each of the three frameworks,
plus the shared Naive baseline, giving 13 methods. The full accuracy table retains all reconciliation methods.

In [ ]:
display(tables['accuracy_by_level'])
print('MCB MASE shape:', tables['mcb_mase'].shape)
display(tables['mcb_mase'].head())

## Training loss and timing

Figures use logs and timings saved by the source experiments, rather than runtimes measured on the current computer.
HFF/HFFPS round counts refer to cloud rounds.

In [ ]:
figure = plot_training(tables, OUTPUT_DIR)
plt.close(figure)
display(Image(filename=str(OUTPUT_DIR / 'training_loss.png')))
display(tables['timings'])
figure = plot_timing(tables, OUTPUT_DIR)
plt.close(figure)
display(Image(filename=str(OUTPUT_DIR / 'training_time_per_round.png')))

## Hierarchy consistency

Compute parent-child discrepancies from structural levels. Normalize MASI by first-difference training-target scales and weight parents equally.
Also report discrepancies normalized by forecast and error scales. Report numerical consistency and forecast accuracy separately.

In [ ]:
coherence = tables['coherence_by_level']
display(coherence.loc[coherence.method_internal.isin(['base','bu','mint_shrinkage','mint_var','naive']),
    ['horizon','source_system','method_internal','level','incoherence_total','MASI',
     'average_scale_free_incoherence_ratio_pct','average_error_scaled_incoherence_ratio_pct']])

In [ ]:
validation = json.loads((OUTPUT_DIR / 'validation.json').read_text())
display(pd.DataFrame([validation]))
print('Source fingerprints:', OUTPUT_DIR / 'source_manifest.json')
print('Generated tables:', sorted(p.name for p in OUTPUT_DIR.glob('*.csv')))